In [3]:
import csv
import os
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


USERNAME = "your_instagram_username"
MAX_ITEMS = 10
OUTPUT_FILE = "instagram_research_data.csv"

CSV_FIELDS = [
    "type",
    "url",
    "caption",
    "hashtags",
    "mentions",
    "timestamp",
    "location"
]


def create_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    return driver


def login_manually(driver):
    driver.get("https://www.instagram.com/accounts/login/")

    print("Please log in manually in the Chrome window.")
    print("After logging in, wait until Instagram home page loads.")
    print("The script will continue automatically in 60 seconds.")

    time.sleep(60)


def create_csv(output_file):
    with open(output_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=CSV_FIELDS)
        writer.writeheader()


def append_row_to_csv(row, output_file):
    with open(output_file, "a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=CSV_FIELDS)
        writer.writerow(row)


def collect_links_from_page(driver, page_url, max_items, existing_links):
    driver.get(page_url)

    wait = WebDriverWait(driver, 20)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    links = []
    last_height = 0
    same_height_count = 0

    while len(existing_links) + len(links) < max_items:
        anchors = driver.find_elements(By.TAG_NAME, "a")

        for anchor in anchors:
            href = anchor.get_attribute("href")

            if not href:
                continue

            if "/p/" in href or "/reel/" in href:
                clean_url = href.split("?")[0]

                if clean_url not in existing_links:
                    existing_links.add(clean_url)
                    links.append(clean_url)

                    if len(existing_links) >= max_items:
                        break

        print(f"Collected total {len(existing_links)} links")

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            same_height_count += 1
        else:
            same_height_count = 0

        if same_height_count >= 3:
            break

        last_height = new_height

    return links


def collect_post_and_reel_links(driver, username, max_items):
    existing_links = set()
    all_links = []

    pages_to_scan = [
        f"https://www.instagram.com/{username}/",
        f"https://www.instagram.com/{username}/reels/"
    ]

    for page_url in pages_to_scan:
        if len(existing_links) >= max_items:
            break

        print(f"Scanning: {page_url}")

        new_links = collect_links_from_page(
            driver=driver,
            page_url=page_url,
            max_items=max_items,
            existing_links=existing_links
        )

        all_links.extend(new_links)

    return all_links[:max_items]


def get_meta_content(driver, selector):
    try:
        element = driver.find_element(By.CSS_SELECTOR, selector)
        return element.get_attribute("content") or ""
    except Exception:
        return ""


def clean_caption_from_meta(meta_description):
    text = meta_description.strip()

    # Instagram metadata often looks like:
    # '123 likes, 4 comments - username on January 1, 2026: "Caption here"'
    match = re.search(r':\s*"(.+)"$', text)

    if match:
        return match.group(1).strip()

    return text


def extract_caption(driver):
    possible_texts = []

    meta_description = get_meta_content(driver, "meta[property='og:description']")
    if meta_description:
        possible_texts.append(clean_caption_from_meta(meta_description))

    meta_title = get_meta_content(driver, "meta[property='og:title']")
    if meta_title:
        possible_texts.append(clean_caption_from_meta(meta_title))

    selectors = [
        "article h1",
        "h1",
        "article span"
    ]

    for selector in selectors:
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)

            for element in elements:
                text = element.text.strip()

                if text and len(text) > 5:
                    possible_texts.append(text)

        except Exception:
            continue

    for text in possible_texts:
        if text:
            return text

    return ""


def extract_timestamp(driver):
    try:
        time_element = driver.find_element(By.TAG_NAME, "time")
        timestamp = time_element.get_attribute("datetime")

        if timestamp:
            return timestamp

    except Exception:
        pass

    return ""


def extract_location(driver):
    try:
        links = driver.find_elements(By.TAG_NAME, "a")

        for link in links:
            href = link.get_attribute("href")
            text = link.text.strip()

            if href and "/explore/locations/" in href and text:
                return text

    except Exception:
        pass

    return ""


def scrape_item_data(driver, url):
    driver.get(url)

    wait = WebDriverWait(driver, 20)

    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    except Exception:
        pass

    time.sleep(3)

    caption = extract_caption(driver)
    hashtags = re.findall(r"#\w+", caption)
    mentions = re.findall(r"@\w+", caption)

    timestamp = extract_timestamp(driver)
    location = extract_location(driver)

    if not caption and not timestamp:
        print("Warning: no caption/timestamp found.")
        print(f"Current URL: {driver.current_url}")
        print(f"Page title: {driver.title}")

    return {
        "caption": caption,
        "hashtags": ", ".join(hashtags),
        "mentions": ", ".join(mentions),
        "timestamp": timestamp,
        "location": location
    }


def main():
    driver = create_driver()

    try:
        create_csv(OUTPUT_FILE)

        login_manually(driver)

        links = collect_post_and_reel_links(
            driver=driver,
            username=USERNAME,
            max_items=MAX_ITEMS
        )

        print(f"Total links found: {len(links)}")

        for index, url in enumerate(links, start=1):
            print(f"Scraping {index}/{len(links)}: {url}")

            item_type = "reel" if "/reel/" in url else "post"
            data = scrape_item_data(driver, url)

            row = {
                "type": item_type,
                "url": url,
                "caption": data["caption"],
                "hashtags": data["hashtags"],
                "mentions": data["mentions"],
                "timestamp": data["timestamp"],
                "location": data["location"]
            }

            append_row_to_csv(row, OUTPUT_FILE)

            print("Saved this item to CSV")

            time.sleep(4)

        print(f"Done. Saved results to {OUTPUT_FILE}")

    except KeyboardInterrupt:
        print("Stopped by user. Already scraped rows are saved in the CSV.")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

Please log in manually in the Chrome window.
After logging in, wait until Instagram home page loads.
The script will continue automatically in 60 seconds.
Scanning: https://www.instagram.com/your_instagram_username/
Collected total 0 links
Collected total 0 links
Collected total 3 links
Collected total 8 links
Scanning: https://www.instagram.com/your_instagram_username/reels/
Collected total 8 links
Collected total 8 links
Collected total 10 links
Total links found: 10
Scraping 1/10: https://www.instagram.com/p/DazmqzAE3HZ/
Saved this item to CSV
Scraping 2/10: https://www.instagram.com/p/DbG5HhjKEBI/
Saved this item to CSV
Scraping 3/10: https://www.instagram.com/p/DakmHabsAiO/
Saved this item to CSV
Scraping 4/10: https://www.instagram.com/p/Dar57-VxtAy/
Saved this item to CSV
Scraping 5/10: https://www.instagram.com/p/DZhgwPUT1cx/
Saved this item to CSV
Scraping 6/10: https://www.instagram.com/p/Da7sYKgzCIJ/
Saved this item to CSV
Scraping 7/10: https://www.instagram.com/p/Da7sYKgzC